# Exploring real epidemic data

Let us explore the COVID-19 time series collected by Johns Hopkins University. We will use **pandas** to select, combine and plot real observations. In the next notebook, we will use the early part of the epidemic to estimate its growth rate.

Some examples are complete; cells marked **Your turn** are for you. Replace `None` and the `TODO` comments with your code, then run the cells in order.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### 1 - What data do we have?

The folder contains three global time series (**confirmed**, **deaths**, **recovered**) and two US time series (**confirmed**, **deaths**, by county).

The values are **cumulative counts**: the total reported up to each date, not the number of new cases that day. Reporting has ended, and recovered data were discontinued before the other indicators: missing or discontinued reporting must not be interpreted as no recoveries.

[Data and file descriptions](https://github.com/CSSEGISandData/COVID-19/tree/master/csse_covid_19_data/csse_covid_19_time_series) · [Reporting notes](https://github.com/CSSEGISandData/COVID-19/blob/master/csse_covid_19_data/README.md)

Use the **Raw** URL to read a CSV rather than its GitHub webpage.

In [ ]:
base_url = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/"

files = {
    "confirmed_global": "time_series_covid19_confirmed_global.csv",
    "deaths_global": "time_series_covid19_deaths_global.csv",
    "recovered_global": "time_series_covid19_recovered_global.csv",
    "confirmed_US": "time_series_covid19_confirmed_US.csv",
    "deaths_US": "time_series_covid19_deaths_US.csv",
}

confirmed = pd.read_csv(base_url + files["confirmed_global"])
confirmed.head()

**Your turn:** load the other four tables. Inspect their first rows with `.head()`.

**Hint:** use the same instruction as above and change the key in `files`.

In [ ]:
deaths = None       # TODO: read the global deaths CSV
recovered = None    # TODO: read the global recovered CSV
confirmed_us = None # TODO: read the US confirmed CSV
deaths_us = None    # TODO: read the US deaths CSV

### 2 - Rows, columns and missing values

In the global tables, a row represents a country or a province/state. The first columns describe the location; the remaining columns are dates. A missing `Province/State` often means that the row represents a whole country.

In [ ]:
print("Rows and columns:", confirmed.shape)
print(confirmed.columns[:8])
confirmed.info()

**Your turn:**

1. How many countries appear in `Country/Region`? Is this equal to the number of rows?
2. Which columns contain missing values? Inspect metadata separately from observations.
3. Compare the global and US columns. Which extra column appears in `deaths_us` but not in `confirmed_us`?

**Hint:** try `.nunique()`, `.unique()`, `.isna().sum()` and `columns.difference(...)`.

In [ ]:
# TODO: count countries and inspect missing values

# TODO: compare the columns in the two US tables

*Write your observations here:*

### 3 - Selecting dates and countries

We only want date columns when adding case counts. Selecting them by their format also prevents us from accidentally adding coordinates or population counts.

In [ ]:
date_cols = confirmed.columns[
    confirmed.columns.str.fullmatch(r"\d{1,2}/\d{1,2}/\d{2}")
]
dates = pd.to_datetime(date_cols, format="%m/%d/%y")

print("First date:", dates.min())
print("Last date:", dates.max())
print("Number of dates:", len(dates))

italy_rows = confirmed.loc[confirmed["Country/Region"] == "Italy"]
italy_rows

**Your turn:** select Australia. How many rows does it have? Print the province/state names.

**Hint:** change the country in the Boolean condition; inspect `Province/State`.

In [ ]:
australia_rows = None  # TODO: select Australia
# TODO: inspect its provinces/states

To obtain national totals, sum the date columns across the rows of a country. This also works when the country has only one row.

In [ ]:
confirmed_by_country = confirmed.groupby("Country/Region")[date_cols].sum(min_count=1)
italy_confirmed = confirmed_by_country.loc["Italy"].copy()
italy_confirmed.index = dates
italy_confirmed = italy_confirmed.sort_index()
italy_confirmed.head()

**Your turn:** extract Australia's national time series and give it a date index. Check one day's total by summing the original Australian rows.

**Hint:** use `.loc["Australia"]` and `.sum(min_count=1)`.

In [ ]:
australia_confirmed = None  # TODO: extract and convert its index
# TODO: compare the national total and the sum of its rows for 3/1/20

### 4 - One table, several indicators

It is easier to compare indicators when **rows are dates** and **columns are variables**. We can extract each country's series and align them by date.

The helper below repeats the steps we have just used. `min_count=1` keeps an entirely missing total missing rather than turning it into zero.

In [ ]:
def country_series(table, country):
    columns = table.columns[table.columns.str.fullmatch(r"\d{1,2}/\d{1,2}/\d{2}")]
    rows = table.loc[table["Country/Region"] == country, columns]
    if rows.empty:
        raise ValueError("Country not found: " + country)
    series = rows.sum(axis=0, min_count=1)
    series.index = pd.to_datetime(series.index, format="%m/%d/%y")
    return series.sort_index()

italy = pd.DataFrame({"confirmed": country_series(confirmed, "Italy")})
italy.head()

**Your turn:** add `deaths` and `recovered` columns to `italy` using `country_series`. Name the index `date` and inspect the first and last rows.

**Hint:** `italy["deaths"] = country_series(deaths, "Italy")`.

In [ ]:
# TODO: add deaths and recovered
# TODO: name the index and inspect both ends of the table

### 5 - From cumulative counts to daily counts

If $C(t)$ is the cumulative number of confirmed cases, the new cases reported on day $t$ are

$$\mathrm{new\ cases}(t)=C(t)-C(t-1).$$

These are **new reported cases**, not the number currently infectious $I(t)$. The first difference is missing because there is no previous observation in the table.

In [ ]:
italy["new_cases"] = italy["confirmed"].diff()
italy[["confirmed", "new_cases"]].head()

**Your turn:**

1. Add `new_deaths` using `.diff()`.
2. Find dates with negative `new_cases` or `new_deaths`.
3. Inspect the surrounding days for one such date, if any exist.

Negative differences can reflect retrospective corrections. Keep them visible; do not automatically replace them with zero.

**Hint:** filter with `italy["new_cases"] < 0` and select a short date interval with `.loc[...]`.

In [ ]:
# TODO: add new_deaths
# TODO: find negative differences and investigate one example

### 6 - Seeing the whole time series

A cumulative curve shows the reported total. Daily counts make waves and reporting fluctuations easier to see.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(italy.index, italy["confirmed"], lw=2)
plt.xlabel("Date")
plt.ylabel("Cumulative confirmed cases")
plt.title("Italy")
plt.xticks(rotation=45);

**Your turn:** plot the full daily case series and add its trailing 7-day mean. Then plot cumulative deaths and recovered counts in separate figures.

**Hint:** `italy["new_cases"].rolling(7, min_periods=7).mean()`.

Does the recovered curve behave like a cumulative count throughout? Look for an abrupt drop or a long run of zeros. These may indicate discontinued reporting, not a biological change. Use the reporting notes before interpreting them.

In [ ]:
# TODO: plot new_cases and its 7-day mean, with a legend

# TODO: plot deaths and recovered separately

*What do the daily and cumulative plots show differently? What limitations do you notice?*

### 7 - Zooming in on the first wave

With a date index, `.loc[start:end]` selects a period including **both endpoints**. Start by looking at the first wave rather than the whole pandemic.

In [ ]:
first_wave = italy.loc["2020-02-20":"2020-05-31"].copy()
print("Days selected:", len(first_wave))
first_wave.head()

**Your turn:** plot `confirmed` and `new_cases` for this interval in separate panels. Find the date of the largest daily count in this interval.

**Hint:** use `plt.subplots(2, 1, figsize=(12, 8))` and `.idxmax()`. A reporting peak is not necessarily the day with the most infections.

In [ ]:
# TODO: plot the two curves
# TODO: find the largest reported daily count and its date

### 8 - Preparing for exponential growth

An exponential curve $y(t)=Ae^{rt}$ becomes a straight line when plotted on a logarithmic vertical axis:

$$\log y(t)=\log A+rt.$$

Here $r$ is a **growth rate**. In the early SIR model, $r\sim eq eta-\gamma$; it is not generally the transmission parameter $eta$.

We will inspect a candidate 14-day window, **24 February–8 March 2020**. This is a teaching choice to examine, not a guaranteed period of constant transmission. Local measures were already in place; nationwide restrictions took effect on 10 March. Reported cases also lag infections.

[National measures and dates](https://www.protezionecivile.gov.it/it/notizia/emergenza-coronavirus--firmato-nuovo-dpcm-con-misure-su-tutto-il-territorio-nazionale/)

In [ ]:
early = italy.loc["2020-02-24":"2020-03-08"].copy()
early["t_days"] = (early.index - early.index[0]).days
print("Observations:", len(early))
early[["t_days", "confirmed", "new_cases"]].head()

**Your turn:**

1. Plot early `confirmed` on linear and logarithmic vertical scales.
2. Repeat for `new_cases`. Before using a log scale, inspect non-positive or missing values; do not add an arbitrary constant to them.
3. Compare the first 7, 10 and 14 observations. Does one straight line seem plausible on the log plot?
4. Extend the plot to 31 March and mark 10 March. Would one exponential describe the whole interval?

**Hint:** `ax.set_yscale("log")`, `early.iloc[:7]`, and `ax.axvline(pd.Timestamp("2020-03-10"), ...)`. We will fit the curves in notebook 3; for now, describe what you see.

In [ ]:
# TODO: compare linear and logarithmic plots

# TODO: compare window lengths and extend the plot into March

*Which interval would you investigate in notebook 3, and why? What could change the growth of reported cases besides transmission?*

### 9 - A dataset for the next notebook

Keep the complete daily table and a small candidate fitting table. Preserve raw observations; the rolling mean is only a visual guide here.

**Your turn:** check that dates are unique, sorted and daily; check missing and non-positive values in the selected fitting columns. Save the two tables with their date index.

**Hint:** `.index.is_unique`, `.index.is_monotonic_increasing`, `.index.to_series().diff()`, `.isna().sum()`, and `.to_csv(..., index_label="date")`.

Use the filenames `italy_timeseries.csv` and `italy_early_stage.csv`. The second table should contain `t_days`, `confirmed`, and `new_cases`.

In [ ]:
# TODO: check the dates and observations

# TODO: save italy to italy_timeseries.csv
# TODO: save early[["t_days", "confirmed", "new_cases"]] to italy_early_stage.csv

### 10 - Optional challenge: US counties

Use the US tables to build a time series for **North Carolina**. Here the state is stored in `Province_State` and the counties in `Admin2`.

**Your turn:** select the state's counties, identify only the date columns, sum them, and plot daily cases and deaths in separate panels. Check that `Population` never enters your time series.

**Hint:** adapt the country example using the US column names; identify date columns separately in each table.

In [ ]:
# TODO: aggregate North Carolina cases and deaths
# TODO: convert the dates, calculate differences and plot

Before moving on: can you explain the difference between cumulative confirmed cases, daily new cases and currently infectious individuals? Which of these do our plots actually show?